## MS Project Automation Test

In [9]:
import os

PATH = r"E:\3 UTHP\1 PMC\2 Construction Programme\P6 to Project"
TEST_FILE_NAME = "Test.mpp"
ACTUAL_FILE_NAME = "UTHP-latest - Copy.mpp"

TEST_FILE_PATH = os.path.join(PATH, TEST_FILE_NAME)
ACTUAL_FILE_PATH = os.path.join(PATH, ACTUAL_FILE_NAME)

In [13]:
import win32com.client
import os


class MSProjectManager:
    """A class to manage Microsoft Project via COM interface."""

    def __init__(self):
        """Initializes the connection to MS Project."""
        try:
            # Connect to an existing instance or start a new one
            self.mpp = win32com.client.Dispatch("MSProject.Application")
            self.mpp.Visible = True  # Make the application visible
            print("Successfully connected to MS Project.")
        except Exception as e:
            print(
                f"Failed to connect to MS Project. Ensure it is installed. Error: {e}"
            )
            self.mpp = None

    def create_new_project(self):
        """Creates a blank new project."""
        if self.mpp:
            self.mpp.FileNew()
            print("New project created.")

    def open_project(self, file_path):
        """Opens an existing .mpp file."""
        if self.mpp and os.path.exists(file_path):
            self.mpp.FileOpen(file_path)
            print(f"Opened project: {file_path}")
        else:
            print(f"File not found: {file_path}")

    def add_activity(self, name, duration=None, start_date=None, resource=None):
        """
        Adds a new activity (task) to the active project.

        :param name: String, the name of the task.
        :param duration: String, e.g., "5d", "2w", "10h".
        :param start_date: String, e.g., "10/25/2023" (MM/DD/YYYY format recommended).
        :param resource: String, name of the resource assigned to the task.
        """
        if not self.mpp or not self.mpp.ActiveProject:
            print("No active project. Please create or open a project first.")
            return None

        try:
            project = self.mpp.ActiveProject

            # Add the task to the bottom of the task list
            task = project.Tasks.Add(name)

            # Set optional properties
            if duration:
                task.Duration = duration
            if start_date:
                task.Start = start_date
            if resource:
                task.ResourceNames = resource

            print(
                f"Activity added: '{name}' (Duration: {duration}, Start: {start_date})"
            )
            return task

        except Exception as e:
            print(f"Error adding activity '{name}': {e}")
            return None
    
    def add_lag_to_fs_predecessors(self):
        """
        Reads the predecessors of each activity. 
        Where the relationship is Finish-to-Start (FS) with 0 lag, adds a 1-day lag.
        """
        if not self.mpp or not self.mpp.ActiveProject:
            print("No active project.")
            return

        project = self.mpp.ActiveProject
        
        # MS Project Constant for Finish-to-Start link type
        pjFinishToStart = 1 
        
        # MS Project stores lag mathematically in minutes.
        # So 1 Day = (Hours per day) * 60 minutes
        # By doing it mathematically, we avoid locale issues (e.g., "1d" vs "1j" in French)
        one_day_lag = int(project.HoursPerDay * 60)

        print("Scanning dependencies for FS relationships with 0 lag...")
        
        # Iterate through all tasks in the project
        for task in project.Tasks:
            if task:  # Skip blank rows (which return as None)
                try:
                    # Iterate through all dependency links connected to this task
                    for dep in task.TaskDependencies:
                        
                        # TaskDependencies returns BOTH incoming and outgoing links.
                        # We only want to process it if the current task is the Successor ('To').
                        if dep.To.ID == task.ID:
                            
                            # Check if Link is FS (1) and Lag is 0
                            if dep.Type == pjFinishToStart and dep.Lag == 0:
                                
                                # Apply the 1-day lag
                                dep.Lag = one_day_lag
                                print(f" -> Added 1-day lag to Task {task.ID} ('{task.Name}') "
                                      f"from Predecessor {dep.From.ID} ('{dep.From.Name}')")
                except Exception as e:
                    print(f"Could not process dependencies for task ID {task.ID}: {e}")

    def save_project(self, file_path=None):
        """Saves the project. If file_path is provided, does a Save As."""
        if not self.mpp or not self.mpp.ActiveProject:
            return

        try:
            if file_path:
                # Need to convert to absolute path for COM
                abs_path = os.path.abspath(file_path)
                self.mpp.FileSaveAs(abs_path)
                print(f"Project saved to: {abs_path}")
            else:
                self.mpp.FileSave()
                print("Project saved.")
        except Exception as e:
            print(f"Error saving project: {e}")

    def close(self, save_changes=False):
        """Quits the MS Project application."""
        if self.mpp:
            # FileClose parameters: 1 = pjSave, 0 = pjDoNotSave, 2 = pjPromptSave
            save_flag = 1 if save_changes else 0
            self.mpp.FileClose(save_flag)
            self.mpp.Quit()
            print("MS Project closed.")

### Example Usage

Creating a new project and adding few activities to it.

In [5]:
# 1. Instantiate the class
manager = MSProjectManager()

# 2. Create a new blank project
manager.create_new_project()

# 3. Add activities (Tasks)
manager.add_activity(
    name="Project Initiation",
    duration="2d",
    start_date="11/01/2023",
    resource="Project Manager"
)

manager.add_activity(
    name="Requirements Gathering",
    duration="1w", # 1 week
    resource="Business Analyst"
)

manager.add_activity(
    name="System Development",
    duration="14d",
    resource="Developer"
)

# 4. Save the project to your current directory
# Note: MS Project requires absolute paths, which the class handles automatically
manager.save_project("My_Python_Project.mpp")

# 5. Optional: Close the application (Uncomment to actually close MS project)
# manager.close()

Successfully connected to MS Project.
New project created.
Activity added: 'Project Initiation' (Duration: 2d, Start: 11/01/2023)
Activity added: 'Requirements Gathering' (Duration: 1w, Start: None)
Activity added: 'System Development' (Duration: 14d, Start: None)
Project saved to: e:\0_Python\pyp6\ProjectScheduling\My_Python_Project.mpp


# Actual Usage

Connecting to an existing project and editing it.

In [14]:
manager = MSProjectManager()
manager.open_project(TEST_FILE_PATH)
manager.add_lag_to_fs_predecessors()

Successfully connected to MS Project.
Opened project: E:\3 UTHP\1 PMC\2 Construction Programme\P6 to Project\Test.mpp
Scanning dependencies for FS relationships with 0 lag...
 -> Added 1-day lag to Task 2 ('Ts2') from Predecessor 1 ('Ts')
 -> Added 1-day lag to Task 3 ('Ts3') from Predecessor 2 ('Ts2')
